# Query DSL — operator tour

This notebook walks through every operator in the query DSL
against the live corpus. Each section runs a small batch of
queries and tabulates the matching clip counts so you can see
what each operator does.

> New to the DevKit? Start with `01_quickstart.ipynb` for an
> introduction to the dataset and the DSL's place in it.

### Grammar in one screen

| form | meaning |
|---|---|
| `<entity>.<attr> <cmp> <value>` | basic attribute predicate |
| `<entity>(<expr>, …)` | same-entity grouping |
| `and`, `or`, `not` | boolean composition |
| `A while B` | A and B with intersecting intervals |
| `A then(K) B` | B starts during A or within K seconds after |
| `A because_of B` | A's `because_of` edge points at a B |
| `within W: E` | restrict E's time window to W's intervals |

The full specification lives in `docs/query_language.md`.


## Setup


In [1]:
import os
from pathlib import Path

from cascade_av.dataset import CascadeDataset

# --- shared notebook styling -----------------------------------------------
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
    "figure.dpi": 110,
    "savefig.dpi": 110,
})

pd.options.display.max_colwidth = 110
pd.options.display.width = 140
pd.options.display.float_format = "{:.2f}".format


In [2]:
ds = CascadeDataset(Path(os.environ["CASCADE_AV_DATASET_ROOT"]))

def run(queries: list[str]) -> pd.DataFrame:
    """Run a batch of DSL queries and tabulate clip counts."""
    return pd.DataFrame(
        [(q, ds.count(q)) for q in queries],
        columns=["query", "matching clips"],
    )


/home/horde/Projects/av-causal-dataset-tools/.claude/worktrees/agent-a0554613148fb8e88/src/cascade_av/dataset.py:118: UserWarning: 1 clip_id(s) had multiple annotation files; kept first by filename. (Set CASCADE_AV_VERBOSE=1 for per-file detail.)
  self._scan(annotation_paths, annotation_root)


## Attribute predicates

The atomic predicate is `entity.attribute <cmp> value`. The
comparison is `=` for equality and `<`, `<=`, `>`, `>=` for
ordered attributes.


In [3]:
run([
    "agent.type = ped",
    "ego.action = decel",
    "light.color = red",
    "env.lanes >= 2",
])


,query,matching clips
0,agent.type = ped,133
1,ego.action = decel,55
2,light.color = red,97
3,env.lanes >= 2,194


## Hierarchical aliases

Several attribute values are *parent aliases* that expand to a
union of children: `vehicle` (car, truck, bus, …),
`vru` (pedestrian, cyclist), `intersection` (4-way, T-junction,
roundabout, …), `turn` (left, right), `change_lane`
(change-left, change-right).


In [4]:
run([
    "agent.type = vehicle",
    "agent.type = vru",
    "agent.type = cyclist",
    "env.type = intersection",
])


,query,matching clips
0,agent.type = vehicle,313
1,agent.type = vru,144
2,agent.type = cyclist,34
3,env.type = intersection,210


## Set membership

`attr in (a, b, c)` is sugar for an OR over the listed values.


In [5]:
run([
    "ego.action in (stop, yield, decel)",
    "agent.type in (ped, cyclist, animal)",
])


,query,matching clips
0,"ego.action in (stop, yield, decel)",197
1,"agent.type in (ped, cyclist, animal)",144


## Same-entity coupling

Inside an `agent(…)` clause, every constraint applies to the
**same** agent. Free-floating predicates may bind to different
agents — note how the count changes.


In [6]:
run([
    "agent(type = vehicle, pos = front)",
    "agent.type = vehicle and agent.pos = front",
])


,query,matching clips
0,"agent(type = vehicle, pos = front)",303
1,agent.type = vehicle and agent.pos = front,307


## Same-entity attribute comparisons

The right-hand side of a scalar comparison may name **another
attribute on the same entity** — not just a literal. This unlocks
shapes the old attr-to-literal form couldn't reach: a `lane_fork`
that *actually widens* (`out_lanes > lanes`) was not expressible
without dropping into Python post-filtering, because no fixed
literal works across envs with different `num_lanes`.

Two equivalent forms — the entity-clause shorthand and the
qualified `<entity>.<attr>` form — both compile to the same AST:

```
env(out_lanes > lanes)       # inside an env(...) clause
env.out_lanes > env.lanes    # qualified form, outside the clause
```

All six scalar comparators are supported (`=`, `!=`, `>`, `>=`,
`<`, `<=`). `in (...)` continues to take a parenthesized value
set — it does not accept an attribute on the RHS.

**None handling.** When either `num_lanes` or `num_out_lanes` is
missing for a candidate environment, the evaluator substitutes
the other side's value into the `None` slot before comparing —
so the two sides count as **equal**. "Equal" satisfies `=`,
`>=`, and `<=`; it does **not** satisfy `!=`, `>`, or `<`. The
full truth table lives in
[`docs/user/query_language.md`](../docs/user/query_language.md)
§4.2.

**Same-entity only.** Cross-entity references on the RHS
(`env.lanes = agent.amount`) and sub-entity descent
(`agent(action.illegal != action.illegal)`) both raise
`QueryParseError`. A more general cross-entity binding mechanism
is a separate, larger gap; it's tracked as future work, not
promised by this feature.


In [7]:
run([
    # Lane forks that actually widen — more out-lanes than in-lanes.
    "env(type = lane_fork, out_lanes > lanes)",
    # Lane merges that actually narrow — fewer out-lanes than in-lanes.
    "env(type = lane_merge, out_lanes < lanes)",
    # No lane-count change, across every env type. The None-as-equal
    # rule means envs with unpopulated num_out_lanes also match here.
    "env(out_lanes = lanes)",
    # Strict difference on lane_merge — None on either side does NOT
    # match `!=`, so this counts only envs with populated, differing
    # in/out lane counts.
    "env(type = lane_merge, out_lanes != lanes)",
])


,query,matching clips
0,"env(type = lane_fork, out_lanes > lanes)",5
1,"env(type = lane_merge, out_lanes < lanes)",7
2,env(out_lanes = lanes),344
3,"env(type = lane_merge, out_lanes != lanes)",12


## Boolean operators

Standard `and`, `or`, `not` over any sub-expression.


In [8]:
run([
    "agent.type = ped and env.type = crosswalk",
    "ego.action = stop or ego.action = yield",
    "agent.type = ped and not env.type = crosswalk",
])


,query,matching clips
0,agent.type = ped and env.type = crosswalk,92
1,ego.action = stop or ego.action = yield,163
2,agent.type = ped and not env.type = crosswalk,41


## Action-type suffixes

Schema 2.0.0 encodes action flags as parenthesized suffixes on
the `action_type` string — `oxd:Walk (jaywalk)`,
`oxd:Walk (jaywalk, erratic)`, `oxd:MakeARightTurn (unprotected)`,
`fst:Nudge (out of lane: into ego lane)`. The suffix is the
single source of truth; the old `*_flag` side-channel fields
are gone. Match a flag by listing every suffix variant that
contains the token on `action.type`.


In [9]:
run([
    # Pedestrians jaywalking (any speed)
    'agent(type = ped, action.type in ('
    '"oxd:Walk (jaywalk)", "oxd:Walk (jaywalk, erratic)", '
    '"oxd:Run (jaywalk)", "oxd:Run (jaywalk, erratic)"))',
    # Pedestrians moving erratically (any speed)
    'agent.action.type in ('
    '"oxd:Walk (erratic)", "oxd:Walk (jaywalk, erratic)", '
    '"oxd:Run (erratic)", "oxd:Run (jaywalk, erratic)")',
    # Unprotected turns (ego or agent)
    'ego.action.type in ('
    '"oxd:MakeALeftTurn (unprotected)", "oxd:MakeARightTurn (unprotected)", '
    '"fst:MakeAUTurn (unprotected)")',
])


,query,matching clips
0,"agent(type = ped, action.type in (""oxd:Walk (jaywalk)"", ""oxd:Walk (jaywalk, erratic)"", ""oxd:Run (jaywalk)""...",35
1,"agent.action.type in (""oxd:Walk (erratic)"", ""oxd:Walk (jaywalk, erratic)"", ""oxd:Run (erratic)"", ""oxd:Run (...",1
2,"ego.action.type in (""oxd:MakeALeftTurn (unprotected)"", ""oxd:MakeARightTurn (unprotected)"", ""fst:MakeAUTurn...",44


## Temporal — `while`

`A while B` keeps only pairs of matches whose intervals
intersect.


In [10]:
run([
    "agent.type = ped while ego.action = decel",
    "light.color = red while ego.action = stop",
])


,query,matching clips
0,agent.type = ped while ego.action = decel,20
1,light.color = red while ego.action = stop,48


## Temporal — `then(K)`

`A then(K) B` keeps pairs where B starts during A or within
K seconds after A ends. Bare `then` (no parens) defaults to
K = 0 — touch or overlap only.


In [11]:
run([
    "light.color = yellow then(3) ego.action = stop",
    "light.color = green then ego.action = drive",
])


,query,matching clips
0,light.color = yellow then(3) ego.action = stop,6
1,light.color = green then ego.action = drive,76


## Relational — `because_of`

Walks the schema's `because_of` causal edge. Every `AgentAction` and
`EgoAction` carries a `because_of: list[str]` of *cause IDs* — IDs of
the actions, signal states, properties, or traffic objects that caused
it. (Never agent IDs directly — the schema records the agent's *action*
as the cause, not the agent itself.)

In the 04 corpus, ~31% of ego actions and ~8% of agent actions are
causally annotated. Cause-type breakdown:

    ego.action.because_of   → AgentAction (257), TrafficObject (58),
                              SignalState (52), AgentProperty (40), …
    agent.action.because_of → AgentAction (67),  SignalState (60),
                              EgoAction (15),    AgentProperty (15), …

The LHS must yield action matches; the RHS accepts any parenthesized
expression — boolean OR, set membership (`in (...)`), cross-modal
compounds, all parse and evaluate cleanly.


In [12]:
# Atomic causes — one cause type per query.
run([
    # …caused by an agent's ACTION
    'ego.action = decel because_of agent.action.type in ("oxd:Walk", "oxd:Run")',
    "ego.action = decel because_of agent.action.type = fst:DrivingInLane",
    "ego.action = decel because_of agent.action.type = oxd:ChangeLane",
    # …caused by a SIGNAL STATE
    "ego.action = stop because_of light.color = red",
    # …caused by a TRAFFIC OBJECT
    "ego.action = stop because_of obj.type = stop_sign",
    "ego.action = yield because_of obj.type = yield_sign",
    "ego.action = decel because_of obj.type in (cone, barrier, debris)",
])


,query,matching clips
0,"ego.action = decel because_of agent.action.type in (""oxd:Walk"", ""oxd:Run"")",6
1,ego.action = decel because_of agent.action.type = fst:DrivingInLane,9
2,ego.action = decel because_of agent.action.type = oxd:ChangeLane,4
3,ego.action = stop because_of light.color = red,31
4,ego.action = stop because_of obj.type = stop_sign,2
5,ego.action = yield because_of obj.type = yield_sign,1
6,"ego.action = decel because_of obj.type in (cone, barrier, debris)",1


### Compound RHS — set membership and boolean OR

Two forms compose alternatives on the right side. `in (...)` is the
cleaner form for many literals; `(A or B)` lets you mix predicates
across attributes.

In [13]:
run([
    # `in (...)` for a set of alternative action types
    "ego.action = decel because_of agent.action.type in (fst:Stop, fst:DrivingInLane)",
    # `(A or B)` — same idea, equivalent here
    "ego.action = decel because_of (agent.action.type = fst:Stop or agent.action.type = fst:DrivingInLane)",
    # Mixing different attributes on the RHS (color OR color)
    "ego.action = decel because_of (light.color = red or light.color = yellow)",
])


,query,matching clips
0,"ego.action = decel because_of agent.action.type in (fst:Stop, fst:DrivingInLane)",9
1,ego.action = decel because_of (agent.action.type = fst:Stop or agent.action.type = fst:DrivingInLane),9
2,ego.action = decel because_of (light.color = red or light.color = yellow),6


### Compound RHS — cross-modal (mix signal states with traffic objects)

This is where `because_of` earns its keep — you can ask cross-modal
causal questions that no single field on the schema captures.

In [14]:
run([
    # Defensive stop because of red light OR stop sign — same behavior, different cause modality
    "ego.action = stop because_of (light.color = red or obj.type = stop_sign)",
    # Full survey of stop-causing controls
    "ego.action = stop because_of "
    "(light.color = red or obj.type = stop_sign or obj.type = yield_sign)",
])


,query,matching clips
0,ego.action = stop because_of (light.color = red or obj.type = stop_sign),33
1,ego.action = stop because_of (light.color = red or obj.type = stop_sign or obj.type = yield_sign),34


### Compound LHS — group similar effects

The LHS can also be a parenthesized expression. Useful when you want
to treat several effects as one category — e.g. defensive braking =
`(stop or decel)`.

In [15]:
run([
    # The full defensive-braking response to a red light
    "(ego.action = stop or ego.action = decel) because_of light.color = red",
    # Compound LHS + compound RHS — the most expressive form
    "(ego.action = stop or ego.action = decel) because_of "
    "(light.color = red or obj.type = stop_sign)",
])


,query,matching clips
0,(ego.action = stop or ego.action = decel) because_of light.color = red,36
1,(ego.action = stop or ego.action = decel) because_of (light.color = red or obj.type = stop_sign),39


### Causality + scene predicates + window scoping

`because_of` isolates the causal edge; everything else (scene context,
time scoping) composes with `and` / `within`. Together they let you ask
focused questions like "during a pedestrian-present window, what
agent-actions caused ego to decelerate?".

In [16]:
run([
    # Ego decels because of ped walking AND a pedestrian is in a crosswalk —
    # three-way enrichment (causal edge + scene predicate + environment).
    'ego.action = decel because_of agent.action.type in ("oxd:Walk", "oxd:Run") '
    "and agent.type = ped and env.type = crosswalk",
    # During a red light, ego stops because of red (self-consistent annotation)
    "within light.color = red: ego.action = stop because_of light.color = red",
    # During a ped-present window, ego decels because of any agent action
    "within agent.type = ped: ego.action = decel because_of "
    'agent.action.type in ("oxd:Walk", "oxd:Run", fst:Stop, fst:DrivingInLane)',
])


,query,matching clips
0,"ego.action = decel because_of agent.action.type in (""oxd:Walk"", ""oxd:Run"") and agent.type = ped and env.ty...",4
1,within light.color = red: ego.action = stop because_of light.color = red,31
2,"within agent.type = ped: ego.action = decel because_of agent.action.type in (""oxd:Walk"", ""oxd:Run"", fst:St...",8


### Patterns that **don't** match — worth knowing

These are the dead ends. Both parse, both run, both always return 0.

| Pattern | Why it fails |
|---|---|
| `because_of agent.type = X` | Agents are never causal targets directly. Use `because_of agent.action.type = Y`. |
| `because_of agent(type = X, action.type = Y)` | The `agent(...)` clause yields Agent matches whose `id` doesn't appear in `because_of` lists (those hold AgentAction ids). Use `because_of agent.action.type = Y` and add the agent-type predicate as a separate `and` clause. |


## Scoping — `within W: E`

Restricts `E`'s evaluation window to `W`'s intervals. Inside,
`not E` means *E does not happen during W* — useful for
negative scenarios like "during red, ego never stops."


In [17]:
run([
    "within light.color = red: not ego.action = stop",
    "within env.type = crosswalk: agent.type = ped",
])


,query,matching clips
0,within light.color = red: not ego.action = stop,49
1,within env.type = crosswalk: agent.type = ped,87


## Annotator-tagged flags

Some attributes are not derived from kinematics but tagged by
the annotator (e.g. *"the ego could have safely cleared this
yellow"*). They compose with the same DSL.


In [18]:
run([
    "light(color = yellow, ego_in_on_yellow = true)",
    "light(color = yellow, could_have_cleared = true)",
])


,query,matching clips
0,"light(color = yellow, ego_in_on_yellow = true)",21
1,"light(color = yellow, could_have_cleared = true)",3
